# Domain-specific developmental-concern model training

This notebook evaluates research models using **baseline checkpoint + the five baseline questions from the predicted domain**. It is a non-diagnostic developmental-monitoring experiment.

The notebook is development-only: model selection, preprocessing, calibration and thresholds use the frozen 500-participant development partition. It refuses to continue if the private dataset or split manifest fails its recorded checksum. It does not evaluate the 125-participant locked-test partition.

Private inputs must remain in the temporary execution environment. Do not commit, publish, display or place participant rows or identifiers in notebook output.

## 1. Install the pinned research environment

Execute this cell once to install the pinned dependencies. Restart the Python runtime if required.

In [ ]:
import sys, subprocess
PACKAGES = [
    'numpy==2.2.2', 'scipy==1.15.1', 'pandas==2.2.3',
    'scikit-learn==1.6.1', 'matplotlib==3.10.0', 'joblib==1.4.2',
    'xgboost==3.2.0', 'catboost==1.2.8', 'lightgbm==4.6.0',
    'interpret==0.7.8', 'imodels==2.0.0'
]
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *PACKAGES])
print('Pinned dependencies installed. Restart the runtime if requested.')

## 2. Configure private persistent progress

Run this cell at the beginning of every hosted session. Choose `RUN_MODE = 'new'` only for the first session and `RUN_MODE = 'resume'` after a disconnection. A new run refuses to use an existing directory, and a resumed run refuses inconsistent or damaged progress. The private dataset is never copied into this directory.

In [ ]:
from pathlib import Path
RUN_MODE = 'new'  # After a disconnection, change this to 'resume'.
CHECKPOINT_ROOT = Path('/content/drive/MyDrive/private-research-checkpoints/domain-specific-v1')
if Path('/content').exists():
    from google.colab import drive, files
    drive.mount('/content/drive')
if RUN_MODE not in {'new', 'resume'}:
    raise ValueError("RUN_MODE must be 'new' or 'resume'.")
if RUN_MODE == 'new' and CHECKPOINT_ROOT.exists() and any(CHECKPOINT_ROOT.iterdir()):
    raise FileExistsError(f'New run refused because checkpoint directory is not empty: {CHECKPOINT_ROOT}')
if RUN_MODE == 'resume' and not CHECKPOINT_ROOT.exists():
    raise FileNotFoundError(f'Resume requested but checkpoint directory does not exist: {CHECKPOINT_ROOT}')
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
print({'run_mode': RUN_MODE, 'checkpoint_root': str(CHECKPOINT_ROOT)})

## 3. Imports, immutable protocol and feature contracts

PR-AUC is the primary model-selection metric. All candidates use the same shared 5-fold, 5-repeat development allocations. The three targets are evaluated separately; overall concern is never trained as a fourth model.

In [ ]:
from pathlib import Path
from dataclasses import dataclass
from hashlib import sha256
from importlib.metadata import version
import json, math, os, platform, sys, time, warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from interpret.glassbox import ExplainableBoostingClassifier
from imodels import FIGSClassifier

try:
    from training.experiments.domain_specific_checkpointing import (CheckpointError, atomic_joblib, atomic_json, fingerprint, load_json, load_result, sha256_file, validate_provenance, write_result)
except ImportError:
    if not Path('/content').exists(): raise
    print('Upload domain_specific_checkpointing.py from training/experiments/.')
    uploaded_helper = files.upload()
    helper = Path('/content/domain_specific_checkpointing.py')
    if helper.name not in uploaded_helper: raise FileNotFoundError('Required checkpoint helper was not uploaded.')
    sys.path.insert(0, '/content')
    from domain_specific_checkpointing import (CheckpointError, atomic_joblib, atomic_json, fingerprint, load_json, load_result, sha256_file, validate_provenance, write_result)

SEED = 20260822
CV_SEED = 20260821
N_SPLITS, N_REPEATS, INNER_SPLITS = 5, 5, 3
FOLD_SEARCH_ITERATIONS = 250
EXPECTED_DATASET_SHA256 = '464A0CD22DE10A425AAA2921D862DBC040584EB8CC51C27140B3AB5946C419B3'
EXPECTED_MANIFEST_SHA256 = '97E62CFD4F2DDBABD59778D1D0508C3B4EDB77E87311FB3F03C31D95AFBC23A5'
CHECKPOINTS = ['CP2M','CP4M','CP6M','CP9M','CP12M','CP15M','CP18M','CP24M','CP30M','CP36M','CP48M','CP60M']
TARGETS = {
    'communication': 'target_communication_concern',
    'cognitive': 'target_cognitive_concern',
    'social_emotional': 'target_social_emotional_concern',
}
FEATURES = {domain: ['baseline_checkpoint_id', *[f'{domain}_q{i}' for i in range(1, 6)]] for domain in TARGETS}
ALL_BASELINE_ITEMS = [f'{domain}_q{i}' for domain in TARGETS for i in range(1, 6)]
assert all(len(values) == 6 for values in FEATURES.values())
print(json.dumps({'targets': TARGETS, 'feature_contracts': FEATURES, 'primary_metric': 'average_precision_pr_auc'}, indent=2))

## 4. Load private inputs

When running inside the repository, the notebook discovers the files locally. In a hosted notebook environment, upload the processed CSV and frozen `split_manifest.csv` only to the temporary private runtime. The notebook does not invent a replacement split.

In [ ]:
def find_repository_file(relative_path):
    relative_path = Path(relative_path)
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    return None

DATASET_PATH = find_repository_file('data/processed/survey-ml-cross-checkpoint-classification.csv')
SPLIT_PATH = find_repository_file('artifacts/splits/split_manifest.csv')
if DATASET_PATH is None or SPLIT_PATH is None:
    try:
        from google.colab import files
        print('Upload the private processed CSV and split_manifest.csv. Do not publish them.')
        uploaded = files.upload()
        for name in uploaded:
            path = Path('/content') / name
            if name == 'survey-ml-cross-checkpoint-classification.csv': DATASET_PATH = path
            if name == 'split_manifest.csv': SPLIT_PATH = path
    except ImportError:
        pass
if DATASET_PATH is None or not Path(DATASET_PATH).is_file():
    raise FileNotFoundError('Private processed dataset is unavailable.')
if SPLIT_PATH is None or not Path(SPLIT_PATH).is_file():
    raise FileNotFoundError('Frozen split manifest is unavailable; no replacement will be generated.')
DATASET_PATH, SPLIT_PATH = Path(DATASET_PATH), Path(SPLIT_PATH)
print('Private inputs located; participant-level contents will not be displayed.')

## 4. Fail-closed validation and development-only projection

This cell validates provenance, schema, values, target derivation and participant allocation before constructing development matrices. Test features and outcomes are deliberately not retained by this notebook.

In [ ]:
dataset_hash = sha256(DATASET_PATH.read_bytes()).hexdigest().upper()
manifest_hash = sha256(SPLIT_PATH.read_bytes()).hexdigest().upper()
assert dataset_hash == EXPECTED_DATASET_SHA256, 'Dataset checksum mismatch.'
assert manifest_hash == EXPECTED_MANIFEST_SHA256, 'Split-manifest checksum mismatch.'
df = pd.read_csv(DATASET_PATH)
manifest = pd.read_csv(SPLIT_PATH, dtype={'participant_number': str})
assert df.shape == (625, 66)
assert list(manifest.columns) == ['participant_number', 'partition']
assert len(manifest) == 625 and manifest['participant_number'].is_unique
assert set(manifest['partition']) == {'development', 'final_test'}
assert df['participant_number'].notna().all() and df['participant_number'].is_unique
assert df['baseline_assessment_id'].notna().all() and df['baseline_assessment_id'].is_unique
assert set(df['baseline_checkpoint_id']) <= set(CHECKPOINTS)
assert df[ALL_BASELINE_ITEMS].notna().all().all()
assert all(set(df[column].astype(int).unique()) <= {0, 1, 2} for column in ALL_BASELINE_ITEMS)
assert all(set(df[target].astype(int).unique()) <= {0, 1} for target in TARGETS.values())
score_columns = {'communication': 'followup_communication_score', 'cognitive': 'followup_cognitive_score', 'social_emotional': 'followup_social_emotional_score'}
for domain, target in TARGETS.items():
    assert ((df[score_columns[domain]].astype(float) <= 50).astype(int) == df[target].astype(int)).all()
assert (df[list(TARGETS.values())].max(axis=1).astype(int) == df['target_any_developmental_concern'].astype(int)).all()
work = df.copy()
work['participant_number'] = work['participant_number'].astype(str)
work = work.merge(manifest, on='participant_number', how='left', validate='one_to_one')
assert work['partition'].notna().all()
assert int(work['partition'].eq('development').sum()) == 500
assert int(work['partition'].eq('final_test').sum()) == 125
development = work.loc[work['partition'].eq('development')].copy()
# Drop the combined frame so this notebook cannot accidentally evaluate the locked test.
del work, df, manifest
X_DEV = {domain: development[columns].copy() for domain, columns in FEATURES.items()}
Y_DEV = {domain: development[target].astype(int).to_numpy() for domain, target in TARGETS.items()}
participant_count = len(development)
positive_counts = {domain: int(values.sum()) for domain, values in Y_DEV.items()}
del development
print({'validated': True, 'development_participants': participant_count, 'locked_test_loaded_for_modeling': False, 'positive_counts': positive_counts})

## 5. Shared repeated development folds

The same allocations are used for all domains, feature contracts and candidates. Combined three-target labels are used only for stratified allocation, never as model inputs.

In [ ]:
from collections import Counter

combined_labels = np.array([''.join(str(Y_DEV[d][i]) for d in TARGETS) for i in range(participant_count)])
checkpoint_values = X_DEV['communication']['baseline_checkpoint_id'].astype(str).to_numpy()
assert min(Counter(combined_labels).values()) >= N_SPLITS
assert all(X_DEV[d]['baseline_checkpoint_id'].astype(str).to_numpy().tolist() == checkpoint_values.tolist() for d in TARGETS)
placeholder = np.zeros(participant_count, dtype=np.int8)

def checkpoint_balance_objective(validation_folds, checkpoints):
    totals = Counter(checkpoints)
    score = 0.0
    for validation_idx in validation_folds:
        observed = Counter(checkpoints[validation_idx])
        for checkpoint, total in totals.items():
            expected = total / len(validation_folds)
            count = observed[checkpoint]
            score += abs(count - expected) / max(expected, 1.0)
            if count == 0:
                score += 100.0
    return score

def best_repeat_split(base_seed):
    best_seed, best_score, best_splits = None, math.inf, None
    for offset in range(FOLD_SEARCH_ITERATIONS):
        candidate_seed = base_seed + offset
        splitter = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=candidate_seed)
        candidate_splits = list(splitter.split(placeholder, combined_labels))
        score = checkpoint_balance_objective([va for _, va in candidate_splits], checkpoint_values)
        if score < best_score:
            best_seed, best_score, best_splits = candidate_seed, score, candidate_splits
    assert best_splits is not None
    return best_seed, best_score, best_splits

FOLDS, allocation_seeds, allocation_scores = [], [], []
validation_coverage = np.zeros((N_REPEATS, participant_count), dtype=np.int8)
for repeat in range(N_REPEATS):
    seed, score, repeat_splits = best_repeat_split(CV_SEED + repeat * FOLD_SEARCH_ITERATIONS)
    allocation_seeds.append(seed); allocation_scores.append(score)
    for train_idx, valid_idx in repeat_splits:
        assert not np.intersect1d(train_idx, valid_idx).size
        for domain in TARGETS:
            assert np.unique(Y_DEV[domain][train_idx]).size == 2
            assert np.unique(Y_DEV[domain][valid_idx]).size == 2
        validation_coverage[repeat, valid_idx] += 1
        FOLDS.append((train_idx, valid_idx))
assert len(FOLDS) == N_SPLITS * N_REPEATS
assert np.all(validation_coverage == 1)
fold_payload = {'allocation_seeds': allocation_seeds, 'folds': [(a.tolist(), b.tolist()) for a, b in FOLDS]}
fold_hash = sha256(json.dumps(fold_payload, separators=(',', ':'), sort_keys=True).encode()).hexdigest().upper()
print({'folds': len(FOLDS), 'allocation_seeds': allocation_seeds, 'checkpoint_balance_scores': allocation_scores, 'fold_plan_sha256': fold_hash})

## 6. Candidate definitions and fold-local pipelines

Checkpoint encoding and any scaling are fitted inside each training fold. Candidate configurations are fixed before results are inspected.

In [ ]:
@dataclass(frozen=True)
class Candidate:
    name: str
    estimator_factory: object
    scale_questions: bool = False
    balanced_sample_weight: bool = False

def candidates():
    return [
        Candidate('logistic_l2_balanced', lambda seed: LogisticRegression(C=1.0, penalty='l2', class_weight='balanced', solver='liblinear', max_iter=2000, random_state=seed), True),
        Candidate('svm_rbf_balanced', lambda seed: SVC(C=1.0, kernel='rbf', gamma='scale', class_weight='balanced', probability=True, random_state=seed), True),
        Candidate('random_forest_balanced_leaf5', lambda seed: RandomForestClassifier(n_estimators=300, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=seed, n_jobs=-1)),
        Candidate('extra_trees_balanced_leaf5', lambda seed: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=5, max_features='sqrt', class_weight='balanced', random_state=seed, n_jobs=-1)),
        Candidate('xgboost_regularized_balanced', lambda seed: XGBClassifier(objective='binary:logistic', eval_metric='logloss', tree_method='hist', n_estimators=300, learning_rate=.03, max_depth=3, min_child_weight=5, subsample=.8, colsample_bytree=.8, reg_alpha=.1, reg_lambda=5, random_state=seed, n_jobs=-1), balanced_sample_weight=True),
        Candidate('catboost_ordered_balanced', lambda seed: CatBoostClassifier(iterations=300, depth=3, learning_rate=.03, l2_leaf_reg=10, auto_class_weights='Balanced', loss_function='Logloss', verbose=False, random_seed=seed)),
        Candidate('ebm_main_effects_balanced', lambda seed: ExplainableBoostingClassifier(interactions=0, max_rounds=3000, min_samples_leaf=8, validation_size=0.15, random_state=seed, n_jobs=-1)),
        Candidate('lightgbm_balanced', lambda seed: LGBMClassifier(n_estimators=300, learning_rate=.03, num_leaves=7, max_depth=3, min_child_samples=20, reg_lambda=5, class_weight='balanced', random_state=seed, n_jobs=-1, verbosity=-1)),
        Candidate('histgradientboosting_balanced', lambda seed: HistGradientBoostingClassifier(max_iter=300, learning_rate=.05, max_leaf_nodes=7, max_depth=3, min_samples_leaf=20, l2_regularization=5, early_stopping=False, class_weight='balanced', random_state=seed)),
        Candidate('figs_balanced_rules8', lambda seed: FIGSClassifier(max_rules=8, max_trees=3, max_depth=3, class_weight='balanced', random_state=seed, n_jobs=-1, verbose=0)),
    ]

def build_pipeline(candidate, domain, seed):
    question_columns = FEATURES[domain][1:]
    preprocess = ColumnTransformer([
        ('checkpoint', OneHotEncoder(categories=[CHECKPOINTS], drop='first', handle_unknown='ignore', sparse_output=False), ['baseline_checkpoint_id']),
        ('questions', StandardScaler() if candidate.scale_questions else 'passthrough', question_columns),
    ], remainder='drop', verbose_feature_names_out=False)
    return Pipeline([('preprocess', preprocess), ('estimator', candidate.estimator_factory(seed))])

def fit_pipeline(candidate, domain, X, y, seed):
    model = build_pipeline(candidate, domain, seed)
    fit_params = {}
    if candidate.balanced_sample_weight:
        fit_params['estimator__sample_weight'] = compute_sample_weight('balanced', y)
    model.fit(X, y, **fit_params)
    return model

CANDIDATES = candidates()
assert len(CANDIDATES) == 10 and len({c.name for c in CANDIDATES}) == 10
print([candidate.name for candidate in CANDIDATES])

temporary_files = list(CHECKPOINT_ROOT.rglob('.*.tmp'))
if temporary_files:
    raise CheckpointError(f'Abandoned temporary checkpoints require review: {temporary_files}')
PACKAGE_NAMES = ['numpy','pandas','scikit-learn','xgboost','catboost','lightgbm','interpret','imodels','joblib']
candidate_configuration = {c.name: repr(c.estimator_factory(SEED).get_params(deep=False)) for c in CANDIDATES}
PROVENANCE = {
    'schema': 'domain-specific-training-provenance-v1',
    'dataset': {'sha256': dataset_hash, 'rows': 625, 'columns': 66},
    'split': {'manifest_sha256': manifest_hash, 'development_rows': 500, 'locked_test_rows': 125},
    'features': FEATURES, 'targets': TARGETS, 'checkpoint_categories': CHECKPOINTS,
    'fold_plan_sha256': fold_hash, 'allocation_seeds': allocation_seeds,
    'configuration': {'seed': SEED, 'cv_seed': CV_SEED, 'n_splits': N_SPLITS, 'n_repeats': N_REPEATS,
                      'inner_splits': INNER_SPLITS, 'fold_search_iterations': FOLD_SEARCH_ITERATIONS,
                      'primary_metric': 'average_precision_pr_auc', 'candidates': candidate_configuration},
    'environment': {'python': platform.python_version(), **{name: version(name) for name in PACKAGE_NAMES}},
    'privacy': {'dataset_persisted': False, 'participant_identifiers_persisted': False,
                'raw_responses_persisted': False, 'target_rows_persisted': False, 'locked_cohort_persisted': False},
}
PROVENANCE_HASH = fingerprint(PROVENANCE)
provenance_path = CHECKPOINT_ROOT / 'experiment-manifest.json'
if RUN_MODE == 'new':
    atomic_json(provenance_path, PROVENANCE)
else:
    validate_provenance(load_json(provenance_path), PROVENANCE)
print({'provenance_verified': True, 'provenance_sha256': PROVENANCE_HASH})

## 7. Metrics and development cross-validation

This is the main training cell. It fits ten individual candidate families for three targets over 25 shared outer folds and stores development-only out-of-fold probabilities in memory.

In [ ]:
def metrics(y, p, threshold=0.5):
    y, p = np.asarray(y, int), np.asarray(p, float)
    decision = (p >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, decision, labels=[0, 1]).ravel()
    return {
        'average_precision_pr_auc': float(average_precision_score(y, p)),
        'roc_auc': float(roc_auc_score(y, p)),
        'sensitivity': float(recall_score(y, decision, zero_division=0)),
        'specificity': float(tn / (tn + fp)) if tn + fp else 0.0,
        'precision': float(precision_score(y, decision, zero_division=0)),
        'negative_predictive_value': float(tn / (tn + fn)) if tn + fn else 0.0,
        'f1': float(f1_score(y, decision, zero_division=0)),
        'balanced_accuracy': float(balanced_accuracy_score(y, decision)),
        'accuracy': float(accuracy_score(y, decision)),
        'brier_score': float(brier_score_loss(y, p)),
        'confusion_matrix': {'true_negative': int(tn), 'false_positive': int(fp), 'false_negative': int(fn), 'true_positive': int(tp)},
    }

OOF = {candidate.name: {domain: np.full((N_REPEATS, participant_count), np.nan) for domain in TARGETS} for candidate in CANDIDATES}
start = time.perf_counter()
for candidate_index, candidate in enumerate(CANDIDATES):
    for domain_index, domain in enumerate(TARGETS):
        X, y = X_DEV[domain], Y_DEV[domain]
        for fold_number, (train_idx, valid_idx) in enumerate(FOLDS):
            repeat = fold_number // N_SPLITS
            fold = fold_number % N_SPLITS
            seed = SEED + candidate_index * 100000 + domain_index * 10000 + fold_number
            checkpoint_dir = CHECKPOINT_ROOT / 'individual' / candidate.name / domain / f'repeat-{repeat:02d}-fold-{fold:02d}'
            identity = {'stage': 'individual', 'candidate': candidate.name, 'domain': domain, 'repeat': repeat,
                        'fold': fold, 'seed': seed, 'provenance_sha256': PROVENANCE_HASH, 'fold_plan_sha256': fold_hash}
            probabilities = load_result(checkpoint_dir, identity)
            if probabilities is None:
                model = fit_pipeline(candidate, domain, X.iloc[train_idx], y[train_idx], seed)
                probabilities = model.predict_proba(X.iloc[valid_idx])[:, 1]
                write_result(checkpoint_dir, probabilities, identity)
            if len(probabilities) != len(valid_idx): raise CheckpointError(f'Fold prediction count mismatch: {checkpoint_dir}')
            OOF[candidate.name][domain][repeat, valid_idx] = probabilities
        assert np.isfinite(OOF[candidate.name][domain]).all()
        print(f'completed {candidate.name} / {domain}')
print({'candidate_domain_fits': len(CANDIDATES) * len(TARGETS) * len(FOLDS), 'runtime_seconds': round(time.perf_counter() - start, 2)})

## 8. Development ranking and Logistic–XGBoost hybrid

Individual candidates are ranked by mean repeat-level PR-AUC. The predefined hybrid averages Logistic Regression and XGBoost probabilities; it is evaluated as a separate candidate and is not allowed to inspect locked-test outcomes.

In [ ]:
HYBRID_NAME = 'logistic_xgboost_equal_weight_soft_voting'
HYBRID_MEMBERS = ('logistic_l2_balanced', 'xgboost_regularized_balanced')
EPS = 1e-6
def logit_feature(p):
    p = np.clip(np.asarray(p, float), EPS, 1 - EPS)
    return np.log(p / (1 - p)).reshape(-1, 1)

def fit_sigmoid_calibrator(probabilities, y, seed):
    return LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, random_state=seed).fit(logit_feature(probabilities), y)

def select_threshold(y, p):
    best = None
    for threshold in np.unique(np.clip(p, EPS, 1 - EPS)):
        result = metrics(y, p, float(threshold))
        key = (result['balanced_accuracy'], result['sensitivity'], result['specificity'], -float(threshold))
        if best is None or key > best[0]: best = (key, float(threshold), result)
    return best[1], best[2]

candidate_lookup = {candidate.name: candidate for candidate in CANDIDATES}
OOF[HYBRID_NAME] = {domain: np.full((N_REPEATS, participant_count), np.nan) for domain in TARGETS}
HYBRID_INNER_THRESHOLDS = {domain: [] for domain in TARGETS}
for domain_index, domain in enumerate(TARGETS):
    X, y = X_DEV[domain], Y_DEV[domain]
    for fold_number, (train_idx, valid_idx) in enumerate(FOLDS):
        repeat = fold_number // N_SPLITS
        fold = fold_number % N_SPLITS
        seed = SEED + 500000 + domain_index * 10000 + fold_number * 100
        checkpoint_dir = CHECKPOINT_ROOT / 'hybrid' / domain / f'repeat-{repeat:02d}-fold-{fold:02d}'
        identity = {'stage': 'hybrid', 'candidate': HYBRID_NAME, 'domain': domain, 'repeat': repeat, 'fold': fold,
                    'seed': seed, 'provenance_sha256': PROVENANCE_HASH, 'fold_plan_sha256': fold_hash}
        saved_hybrid = load_result(checkpoint_dir, identity)
        if saved_hybrid is not None:
            if len(saved_hybrid) != len(valid_idx): raise CheckpointError(f'Hybrid prediction count mismatch: {checkpoint_dir}')
            saved_manifest = load_json(checkpoint_dir / 'manifest.json')
            saved_threshold = float(saved_manifest['selected_threshold'])
            if not 0 <= saved_threshold <= 1: raise CheckpointError(f'Hybrid threshold is invalid: {checkpoint_dir}')
            HYBRID_INNER_THRESHOLDS[domain].append(saved_threshold)
            OOF[HYBRID_NAME][domain][repeat, valid_idx] = saved_hybrid
            continue
        inner = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=seed)
        member_inner, member_outer = {}, {}
        for member_offset, member in enumerate(HYBRID_MEMBERS):
            inner_probabilities = np.full(len(train_idx), np.nan)
            outer_train_X, outer_train_y = X.iloc[train_idx], y[train_idx]
            for inner_fold, (inner_train, inner_valid) in enumerate(inner.split(outer_train_X, outer_train_y)):
                fitted = fit_pipeline(candidate_lookup[member], domain, outer_train_X.iloc[inner_train], outer_train_y[inner_train], seed + member_offset * 20 + inner_fold)
                inner_probabilities[inner_valid] = fitted.predict_proba(outer_train_X.iloc[inner_valid])[:, 1]
            assert np.isfinite(inner_probabilities).all()
            calibrator = fit_sigmoid_calibrator(inner_probabilities, outer_train_y, seed + member_offset + 50)
            member_inner[member] = calibrator.predict_proba(logit_feature(inner_probabilities))[:, 1]
            fitted_outer = fit_pipeline(candidate_lookup[member], domain, outer_train_X, outer_train_y, seed + member_offset + 70)
            raw_outer = fitted_outer.predict_proba(X.iloc[valid_idx])[:, 1]
            member_outer[member] = calibrator.predict_proba(logit_feature(raw_outer))[:, 1]
        inner_hybrid = 0.5 * member_inner[HYBRID_MEMBERS[0]] + 0.5 * member_inner[HYBRID_MEMBERS[1]]
        outer_hybrid = 0.5 * member_outer[HYBRID_MEMBERS[0]] + 0.5 * member_outer[HYBRID_MEMBERS[1]]
        threshold, _ = select_threshold(y[train_idx], inner_hybrid)
        HYBRID_INNER_THRESHOLDS[domain].append(float(threshold))
        write_result(checkpoint_dir, outer_hybrid, {**identity, 'selected_threshold': float(threshold)})
        OOF[HYBRID_NAME][domain][repeat, valid_idx] = outer_hybrid
    assert np.isfinite(OOF[HYBRID_NAME][domain]).all()

ranking_rows = []
for candidate_name, domain_values in OOF.items():
    for domain, repeated in domain_values.items():
        repeat_metrics = [metrics(Y_DEV[domain], repeated[r]) for r in range(N_REPEATS)]
        ranking_rows.append({
            'candidate': candidate_name, 'domain': domain,
            'mean_pr_auc': float(np.mean([m['average_precision_pr_auc'] for m in repeat_metrics])),
            'sd_pr_auc': float(np.std([m['average_precision_pr_auc'] for m in repeat_metrics], ddof=1)),
            'mean_roc_auc': float(np.mean([m['roc_auc'] for m in repeat_metrics])),
            'mean_brier': float(np.mean([m['brier_score'] for m in repeat_metrics])),
        })
ranking = pd.DataFrame(ranking_rows).sort_values(['domain', 'mean_pr_auc', 'mean_roc_auc'], ascending=[True, False, False])
evaluation_files = sorted([p for namespace in ['individual', 'hybrid'] for p in (CHECKPOINT_ROOT / namespace).rglob('*') if p.is_file()])
evaluation_inventory = fingerprint([(str(p.relative_to(CHECKPOINT_ROOT)), sha256_file(p)) for p in evaluation_files])
ranking_checkpoint = {'schema': 'domain-specific-stage-v1', 'stage': 'ranking',
                      'provenance_sha256': PROVENANCE_HASH, 'dependency_sha256': evaluation_inventory,
                      'rows': json.loads(ranking.to_json(orient='records'))}
ranking_path = CHECKPOINT_ROOT / 'stages' / 'ranking' / 'result.json'
if ranking_path.exists(): validate_provenance(load_json(ranking_path), ranking_checkpoint)
else: atomic_json(ranking_path, ranking_checkpoint)
display(ranking.round(4).style.hide(axis='index'))

## 9. Nested calibration and threshold evaluation for development-selected candidates

Candidate identity is selected from repeated development PR-AUC. For each outer fold, calibration and threshold selection use only inner out-of-fold predictions from the outer-training participants. Outer-validation labels never fit the calibrator or threshold.

In [ ]:
selected_names = {domain: ranking.loc[ranking.domain.eq(domain)].iloc[0].candidate for domain in TARGETS}

def raw_inner_oof(candidate_name, domain, X, y, seed):
    inner = StratifiedKFold(n_splits=INNER_SPLITS, shuffle=True, random_state=seed)
    probabilities = np.full(len(y), np.nan)
    for k, (tr, va) in enumerate(inner.split(X, y)):
        fitted = fit_pipeline(candidate_lookup[candidate_name], domain, X.iloc[tr], y[tr], seed + k)
        probabilities[va] = fitted.predict_proba(X.iloc[va])[:, 1]
    assert np.isfinite(probabilities).all()
    return probabilities

def fit_raw(candidate_name, domain, X_train, y_train, X_predict, seed):
    fitted = fit_pipeline(candidate_lookup[candidate_name], domain, X_train, y_train, seed)
    return fitted, fitted.predict_proba(X_predict)[:, 1]

NESTED = {domain: np.full((N_REPEATS, participant_count), np.nan) for domain in TARGETS}
OUTER_THRESHOLDS = {domain: [] for domain in TARGETS}
for domain_index, domain in enumerate(TARGETS):
    X, y, selected = X_DEV[domain], Y_DEV[domain], selected_names[domain]
    if selected == HYBRID_NAME:
        NESTED[domain][:] = OOF[HYBRID_NAME][domain]
        OUTER_THRESHOLDS[domain] = list(HYBRID_INNER_THRESHOLDS[domain])
        stage_value = {'schema': 'domain-specific-stage-v1', 'stage': 'nested-calibration', 'domain': domain,
                       'candidate': selected, 'provenance_sha256': PROVENANCE_HASH,
                       'dependency_sha256': evaluation_inventory, 'thresholds': OUTER_THRESHOLDS[domain]}
        stage_path = CHECKPOINT_ROOT / 'stages' / 'nested-calibration' / f'{domain}.json'
        if stage_path.exists(): validate_provenance(load_json(stage_path), stage_value)
        else: atomic_json(stage_path, stage_value)
        continue
    for fold_number, (train_idx, valid_idx) in enumerate(FOLDS):
        repeat = fold_number // N_SPLITS
        fold = fold_number % N_SPLITS
        seed = SEED + 700000 + domain_index * 10000 + fold_number * 100
        checkpoint_dir = CHECKPOINT_ROOT / 'stages' / 'nested-calibration' / domain / f'repeat-{repeat:02d}-fold-{fold:02d}'
        identity = {'stage': 'nested-calibration-fold', 'candidate': selected, 'domain': domain, 'repeat': repeat,
                    'fold': fold, 'seed': seed, 'provenance_sha256': PROVENANCE_HASH,
                    'dependency_sha256': evaluation_inventory}
        saved_nested = load_result(checkpoint_dir, identity)
        if saved_nested is not None:
            if len(saved_nested) != len(valid_idx): raise CheckpointError(f'Nested prediction count mismatch: {checkpoint_dir}')
            saved_threshold = float(load_json(checkpoint_dir / 'manifest.json')['selected_threshold'])
            OUTER_THRESHOLDS[domain].append(saved_threshold)
            NESTED[domain][repeat, valid_idx] = saved_nested
            continue
        inner_raw = raw_inner_oof(selected, domain, X.iloc[train_idx], y[train_idx], seed)
        calibrator = LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, random_state=seed).fit(logit_feature(inner_raw), y[train_idx])
        inner_calibrated = calibrator.predict_proba(logit_feature(inner_raw))[:, 1]
        threshold, _ = select_threshold(y[train_idx], inner_calibrated)
        _, outer_raw = fit_raw(selected, domain, X.iloc[train_idx], y[train_idx], X.iloc[valid_idx], seed)
        nested_probabilities = calibrator.predict_proba(logit_feature(outer_raw))[:, 1]
        write_result(checkpoint_dir, nested_probabilities, {**identity, 'selected_threshold': float(threshold)})
        NESTED[domain][repeat, valid_idx] = nested_probabilities
        OUTER_THRESHOLDS[domain].append(float(threshold))
    assert np.isfinite(NESTED[domain]).all()
print({'selected_candidates': selected_names, 'outer_threshold_ranges': {d: [min(v), max(v)] for d, v in OUTER_THRESHOLDS.items()}})

## 10. Freeze final development configurations and fit artifacts

The final calibrator and threshold use averaged repeated out-of-fold development predictions. This is development selection evidence, not independent validation. Final classifier weights are fitted on all 500 development participants.

In [ ]:
OUTPUT_DIR = CHECKPOINT_ROOT / 'artifacts'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
nested_files = sorted(p for p in (CHECKPOINT_ROOT / 'stages' / 'nested-calibration').rglob('*') if p.is_file())
finalization_dependency = fingerprint([(str(p.relative_to(CHECKPOINT_ROOT)), sha256_file(p)) for p in nested_files])
final_summary, release_domains = {}, {}
for domain_index, domain in enumerate(TARGETS):
    selected, X, y = selected_names[domain], X_DEV[domain], Y_DEV[domain]
    artifact = OUTPUT_DIR / f'{domain}.joblib'
    completion_path = CHECKPOINT_ROOT / 'stages' / 'finalization' / f'{domain}.json'
    if completion_path.exists():
        completed = load_json(completion_path)
        expected_completion = {'schema': 'domain-specific-stage-v1', 'stage': 'finalization', 'domain': domain,
                               'candidate': selected, 'provenance_sha256': PROVENANCE_HASH,
                               'dependency_sha256': finalization_dependency}
        for key, value in expected_completion.items():
            if completed.get(key) != value: raise CheckpointError(f'Finalization {key} mismatch for {domain}')
        if not artifact.is_file() or sha256_file(artifact) != completed.get('artifact_sha256'):
            raise CheckpointError(f'Final artifact is missing or corrupted for {domain}')
        final_summary[domain] = completed['development_summary']
        release_domains[domain] = completed['release_domain']
        continue
    calibration_artifact = CHECKPOINT_ROOT / 'stages' / 'final-calibration' / f'{domain}.joblib'
    threshold_path = CHECKPOINT_ROOT / 'stages' / 'threshold-selection' / f'{domain}.json'
    calibration_reused = threshold_path.exists() or calibration_artifact.exists()
    if calibration_reused:
        if not threshold_path.is_file() or not calibration_artifact.is_file():
            raise CheckpointError(f'Incomplete final calibration checkpoint for {domain}')
        threshold_stage = load_json(threshold_path)
        for key, value in {'schema': 'domain-specific-stage-v1', 'stage': 'threshold-selection', 'domain': domain,
                           'candidate': selected, 'provenance_sha256': PROVENANCE_HASH,
                           'dependency_sha256': finalization_dependency}.items():
            if threshold_stage.get(key) != value: raise CheckpointError(f'Threshold-selection {key} mismatch for {domain}')
        if sha256_file(calibration_artifact) != threshold_stage.get('calibration_artifact_sha256'):
            raise CheckpointError(f'Final calibration artifact is corrupted for {domain}')
        calibration_state = joblib.load(calibration_artifact)
        final_calibrator = calibration_state['calibrator']
        calibrated_development = calibration_state['calibrated_development']
        final_threshold = float(threshold_stage['threshold'])
        final_metrics = threshold_stage['development_metrics']
        if selected == HYBRID_NAME:
            final_model = {member: fit_pipeline(candidate_lookup[member], domain, X, y, SEED + 950000 + domain_index * 10 + member_offset)
                           for member_offset, member in enumerate(HYBRID_MEMBERS)}
        else:
            final_model = fit_pipeline(candidate_lookup[selected], domain, X, y, SEED + 950000 + domain_index)
    elif selected == HYBRID_NAME:
        final_model, final_calibrator, calibrated_members = {}, {}, {}
        for member_offset, member in enumerate(HYBRID_MEMBERS):
            member_raw_average = OOF[member][domain].mean(axis=0)
            final_calibrator[member] = fit_sigmoid_calibrator(member_raw_average, y, SEED + 900000 + domain_index * 10 + member_offset)
            calibrated_members[member] = final_calibrator[member].predict_proba(logit_feature(member_raw_average))[:, 1]
            final_model[member] = fit_pipeline(candidate_lookup[member], domain, X, y, SEED + 950000 + domain_index * 10 + member_offset)
        calibrated_development = 0.5 * calibrated_members[HYBRID_MEMBERS[0]] + 0.5 * calibrated_members[HYBRID_MEMBERS[1]]
    else:
        raw_average = OOF[selected][domain].mean(axis=0)
        final_calibrator = fit_sigmoid_calibrator(raw_average, y, SEED + 900000 + domain_index)
        calibrated_development = final_calibrator.predict_proba(logit_feature(raw_average))[:, 1]
        final_model = fit_pipeline(candidate_lookup[selected], domain, X, y, SEED + 950000 + domain_index)
    if not calibration_reused:
        final_threshold, final_metrics = select_threshold(y, calibrated_development)
        calibration_hash = atomic_joblib(calibration_artifact, {'calibrator': final_calibrator,
                                                                 'calibrated_development': calibrated_development})
        atomic_json(threshold_path, {'schema': 'domain-specific-stage-v1', 'stage': 'threshold-selection',
                                     'domain': domain, 'candidate': selected, 'provenance_sha256': PROVENANCE_HASH,
                                     'dependency_sha256': finalization_dependency,
                                     'calibration_artifact_sha256': calibration_hash, 'threshold': final_threshold,
                                     'development_metrics': final_metrics})
    bundle = {
        'domain': domain, 'target': TARGETS[domain], 'feature_columns': FEATURES[domain],
        'candidate': selected, 'model': final_model, 'calibrator': final_calibrator,
        'threshold': final_threshold, 'checkpoint_categories': CHECKPOINTS,
        'dataset_sha256': dataset_hash, 'split_manifest_sha256': manifest_hash,
        'fold_plan_sha256': fold_hash, 'development_participants': participant_count,
    }
    artifact_hash = atomic_joblib(artifact, bundle)
    final_summary[domain] = {'candidate': selected, 'threshold': final_threshold, 'development_metrics': final_metrics}
    release_domains[domain] = {'artifact': artifact.name, 'artifact_sha256': artifact_hash, 'candidate': selected, 'feature_columns': FEATURES[domain], 'threshold': final_threshold}
    atomic_json(completion_path, {'schema': 'domain-specific-stage-v1', 'stage': 'finalization', 'domain': domain,
                                  'candidate': selected, 'provenance_sha256': PROVENANCE_HASH,
                                  'dependency_sha256': finalization_dependency, 'artifact_sha256': artifact_hash,
                                  'development_summary': final_summary[domain], 'release_domain': release_domains[domain]})

release = {
    'schema': 'domain-specific-development-models-v1',
    'status': 'development_frozen_pending_locked_test_evaluation',
    'scope': 'six_conceptual_inputs_per_domain',
    'dataset_sha256': dataset_hash, 'split_manifest_sha256': manifest_hash,
    'fold_plan_sha256': fold_hash, 'development_participants': participant_count,
    'locked_test_evaluated': False, 'primary_metric': 'average_precision_pr_auc',
    'domains': release_domains,
    'runtime_versions': {name: version(name) for name in ['numpy','pandas','scikit-learn','xgboost','catboost','lightgbm','interpret','imodels','joblib']},
    'limitations': ['Research-only and non-diagnostic.', 'No prospective, external or clinical validation.', 'Development selection is not independent test evidence.'],
}
release_path = OUTPUT_DIR / 'release-manifest.json'
summary_path = OUTPUT_DIR / 'development-summary.json'
if release_path.exists(): validate_provenance(load_json(release_path), release)
else: atomic_json(release_path, release)
if summary_path.exists(): validate_provenance(load_json(summary_path), final_summary)
else: atomic_json(summary_path, final_summary)
display(pd.DataFrame([{'domain': d, 'candidate': v['candidate'], 'threshold': v['threshold'], **{k: val for k, val in v['development_metrics'].items() if not isinstance(val, dict)}} for d, v in final_summary.items()]).round(4))
print({'output_directory': str(OUTPUT_DIR), 'locked_test_evaluated': False})

## 11. Export aggregate evidence and frozen artifacts

Download only reviewed artifacts and aggregate reports. The bundle contains fitted research models, so store it according to project governance. It contains no participant identifiers or row-level predictions.

In [ ]:
import shutil
archive_base = Path('/content/domain-specific-models-v1') if Path('/content').exists() else Path('tmp/domain-specific-models-v1')
archive_base.parent.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(archive_base), 'zip', root_dir=OUTPUT_DIR)
print({'archive': archive, 'sha256': sha256(Path(archive).read_bytes()).hexdigest().upper()})
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print('Local execution: archive is available at', archive)

## Required interpretation

These results select research configurations using development data only. They do not establish that the six-input approach is better than the existing sixteen-input approach, and they are not clinical evidence. The separate locked-test notebook may evaluate these frozen artifacts but must not change their candidates, preprocessing, calibration or thresholds.